# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: C:\dev\projects\capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "Jack"  # your name, recorded on every result row (default: OS user)
RESUME        = True  # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce",
                  "Q_heisenberg_pauli", "Q_heisenberg_single_bit", "Q_zz_pauli", "Q_zz_pce", "Q_zz_single_bit"]
ACTIVE_TRACKS = QUANTUM_TRACKS                                 # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["decision_tree", "adaboost", "random_forest", "xgboost",
                 "linear_svm", "sig_kernel", "linear_readout",
                 "rnn", "gru", "lstm"]  # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['Q_su_pce', 'Q_su_pauli', 'Q_su_bitstring', 'Q_su_single_bit', 'Q_heisenberg_pce', 'Q_heisenberg_pauli', 'Q_heisenberg_single_bit', 'Q_zz_pauli', 'Q_zz_pce', 'Q_zz_single_bit']
Models:   ['decision_tree', 'adaboost', 'random_forest', 'xgboost', 'linear_svm', 'sig_kernel', 'linear_readout', 'rnn', 'gru', 'lstm']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track Q_su_pce target INDPRO     loaded   shape=(767, 9)
  Track Q_su_pce target PAYEMS     loaded   shape=(767, 9)
  Track Q_su_pce target CPIAUCSL   loaded   shape=(767, 9)
  Track Q_su_pce target S&P 500    loaded   shape=(767, 9)
  Track Q_su_pauli target INDPRO     loaded   shape=(767, 15)
  Track Q_su_pauli target PAYEMS     loaded   shape=(767, 15)
  Track Q_su_pauli target CPIAUCSL   loaded   shape=(767, 18)
  Track Q_su_pauli target S&P 500    loaded   shape=(767, 18)
  Track Q_su_bitstring target INDPRO     loaded   shape=(767, 32)
  Track Q_su_bitstring target PAYEMS     loaded   shape=(767, 32)
  Track Q_su_bitstring target CPIAUCSL   loaded   shape=(767, 64)
  Track Q_su_bitstring target S&P 500    loaded   shape=(767, 64)
  Track Q_su_single_bit target INDPRO     loaded   shape=(767, 5)
  Track Q_su_single_bit target PAYEMS     loaded   shape=(767, 5)
  Track Q_su_single_bit target CPIAUCSL   loaded   shape=(767, 6)
  Track Q_su_single_bit target S&P 500    loaded   sha

  Track Q_zz_single_bit target PAYEMS     loaded   shape=(767, 11)
  Track Q_zz_single_bit target CPIAUCSL   loaded   shape=(767, 13)
  Track Q_zz_single_bit target S&P 500    loaded   shape=(767, 13)
Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Track Q_su_pce: resuming, 7724 runs already on disk will be skipped


  8724/776400 runs complete · 1m19s elapsed · ~16h50m left


  9724/776400 runs complete · 2m51s elapsed · ~18h13m left


  Saved: raw/decision_tree/Q_su_pce.parquet   new=300  total=300
  Saved: raw/adaboost/Q_su_pce.parquet   new=300  total=300
  Saved: raw/random_forest/Q_su_pce.parquet   new=300  total=300
  Saved: raw/xgboost/Q_su_pce.parquet   new=300  total=300
  Saved: raw/linear_svm/Q_su_pce.parquet   new=300  total=300
  Saved: raw/sig_kernel/Q_su_pce.parquet   new=300  total=300
  Saved: raw/rnn/Q_su_pce.parquet   new=300  total=300
  Saved: raw/gru/Q_su_pce.parquet   new=300  total=300
  Saved: raw/lstm/Q_su_pce.parquet   new=300  total=300


  10724/776400 runs complete · 4m19s elapsed · ~18h20m left


...

  771620/776400 runs complete · 19h50m elapsed · ~7m46s left


  Saved: raw/decision_tree/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/adaboost/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/random_forest/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/xgboost/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/linear_svm/Q_zz_single_bit.parquet   new=224  total=7724


  Saved: raw/sig_kernel/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/linear_readout/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/rnn/Q_zz_single_bit.parquet   new=224  total=7724
  Saved: raw/gru/Q_zz_single_bit.parquet   new=224  total=7724


  Saved: raw/lstm/Q_zz_single_bit.parquet   new=224  total=7724
Loop complete. Total rows: 733780 in 19h51m


Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae  \
model    track              target   horizon                                  
adaboost Q_heisenberg_pauli CPIAUCSL 1        0.000009  0.002195   0.002195   
                                     3        0.000057  0.005282   0.005282   
                                     6        0.000177  0.009400   0.009400   
                            INDPRO   1        0.000096  0.005598   0.005598   
                                     3        0.000387  0.011618   0.011618   
...                                                ...       ...        ...   
xgboost  Q_zz_single_bit    PAYEMS   3        0.000133  0.005300   0.005300   
                                     6        0.000249  0.009272   0.009272   
                            S&P 500  1        0.001528  0.028999   0.028999   
                                     3        0.005787  0.057070   0.057070   
                                     6        0.012958  0.085673   0.085673   

                                                  rmse      mase     rmspe  
model    track              target   horizon                                
adaboost Q_heisenberg_pauli CPIAUCSL 1        0.002195  1.161184  1.746241  
                                     3        0.005282  1.141444  2.411655  
                                     6        0.009400  1.342237  1.400389  
                            INDPRO   1        0.005598  0.892536  6.417678  
                                     3        0.011618  0.996155  4.675681  
...                                                ...       ...       ...  
xgboost  Q_zz_single_bit    PAYEMS   3        0.005300  1.932013  2.444054  
                                     6        0.009272  1.731275  5.349831  
                            S&P 500  1        0.028999  0.877603  3.797841  
                                     3        0.057070  0.823439  2.105983  
                                     6        0.085673  0.855550  1.908518  

[1200 rows x 6 columns]

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model   track           target   horizon usrec                                  
AR6     B               CPIAUCSL 1       0      0.000005  0.001586   0.001586   
                                         1      0.000019  0.003242   0.003242   
                                 3       0      0.000027  0.003785   0.003785   
                                         1      0.000161  0.009629   0.009629   
                                 6       0      0.000093  0.006961   0.006961   
...                                                  ...       ...        ...   
xgboost Q_zz_single_bit S&P 500  1       1      0.003793  0.046751   0.046751   
                                 3       0      0.004647  0.051856   0.051856   
                                         1      0.014571  0.097236   0.097236   
                                 6       0      0.009090  0.074541   0.074541   
                                         1      0.042596  0.170972   0.170972   

                                                    rmse      mase     rmspe  
model   track           target   horizon usrec                                
AR6     B               CPIAUCSL 1       0      0.001586  0.860288  1.348996  
                                         1      0.003242  1.546022  1.333686  
                                 3       0      0.003785  0.823602  2.077020  
                                         1      0.009629  1.925051  1.826281  
                                 6       0      0.006961  1.001412  1.211426  
...                                                  ...       ...       ...  
xgboost Q_zz_single_bit S&P 500  1       1      0.046751  1.326651  1.306572  
                                 3       0      0.051856  0.752689  2.227379  
                                         1      0.097236  1.368405  1.170910  
                                 6       0      0.074541  0.752026  2.015619  
                                         1      0.170972  1.648774  1.090794  

[2976 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,adaboost,Q_heisenberg_pauli,CPIAUCSL,1,0,1972-04-01,0.002407,0.002688
1,adaboost,Q_heisenberg_pauli,CPIAUCSL,1,1,1972-05-01,0.002401,0.003331
2,adaboost,Q_heisenberg_pauli,CPIAUCSL,1,2,1972-06-01,0.002395,0.002874
3,adaboost,Q_heisenberg_pauli,CPIAUCSL,1,3,1972-07-01,0.002389,0.002904
4,adaboost,Q_heisenberg_pauli,CPIAUCSL,1,4,1972-08-01,0.004762,0.002870


mean      sum
model    track                                  
adaboost Q_heisenberg_pauli       0.062  475.064
         Q_heisenberg_pce         0.042  323.327
         Q_heisenberg_single_bit  0.044  342.873
         Q_su_bitstring           0.082  632.016
         Q_su_pauli               0.096  745.030
...                                 ...      ...
xgboost  Q_su_pce                 0.029  220.882
         Q_su_single_bit          0.012   91.242
         Q_zz_pauli               0.024  184.144
         Q_zz_pce                 0.013  102.046
         Q_zz_single_bit          0.014  111.337

[100 rows x 2 columns]